In [1]:
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)

import numpy as np
import pandas as pd
from os import listdir
from os.path import isfile, join
from mhcnames import normalize_allele_name

METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/'  # windows
DATA_PATH_IMMUNOGENIC = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/immunogenic/per_length_per_kfold_split/'  # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/low_iterations_training/"



In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)



per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

per_allele_per_kfold_per_length_immunogenic_train = read_data(DATA_PATH_IMMUNOGENIC, ALLELES, "train", "immunogenic")
per_allele_per_kfold_per_length_immunogenic_test = read_data(DATA_PATH_IMMUNOGENIC, ALLELES, "test", "immunogenic")
per_allele_per_kfold_per_length_non_immunogenic_train = read_data(DATA_PATH_IMMUNOGENIC, ALLELES, "train",
                                                                  "nonimmunogenic")
per_allele_per_kfold_per_length_non_immunogenic_test = read_data(DATA_PATH_IMMUNOGENIC, ALLELES, "test",
                                                                 "nonimmunogenic")

ALLELES

['HLA-A*02:01']

# 2. Read models

In [5]:
from hmm_logic_methods import load_model
total_splits = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]])

selected_models = list()
RUNS= 2
target_length = 9
states = 18
# split_num = 3
# run_num = 1
# states= 18
# selected_binding1 = f"{run_num}_run-{target_length}-{states}-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-{split_num}"
# selected_models.append(selected_binding1)

# split_num = 4
# run_num = 1
# selected_binding2 = f"{run_num}_run-{target_length}-{states}-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-{split_num}"
# selected_models.append(selected_binding2)
per_name_base_model = dict()
binding_models = list()

source_models_path =  "experiment_results/low_iterations_training/"
for run_num in range(RUNS):
    for split_num in range(total_splits):
        binding_models.append(f"{run_num}_run-{target_length}-{states}-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-{split_num}")

for model_name in binding_models:
    model = load_model(source_models_path, model_name)
    per_name_base_model[model_name] = model

source_models_path =  "experiment_results/iterations/"

selected_binding_1 = f"{1}_run-{target_length}-{states}-st-baum-welch-alg-30-min_iters-69-max_iters_model-HLA_A_02_01-{2}"
selected_binding_2 = f"{1}_run-{target_length}-{states}-st-baum-welch-alg-30-min_iters-69-max_iters_model-HLA_A_02_01-{0}"
selected_binding_3 = f"{0}_run-{target_length}-{states}-st-baum-welch-alg-30-min_iters-69-max_iters_model-HLA_A_02_01-{3}"
binding_models = list()
binding_models.append(selected_binding_1)
binding_models.append(selected_binding_2)
binding_models.append(selected_binding_3)

for model_name in binding_models:
    model = load_model(source_models_path, model_name)
    per_name_base_model[model_name] = model



# 3. Training

In [6]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions

## 3.1 Prepare modelsFrom 9-mer binding to 9-mer immunogenic

In [7]:
cutoff = 0.00005
train_size = len(per_allele_per_kfold_per_length_immunogenic_train[ALLELES[0]][0][9])
## Stop threshold Hardcoded here
stop_threshold =  abs(cutoff * train_size * -22)
stop_threshold

0.4378

In [13]:
RUNS = 1
TARGET_LENGTHS = [9]
lengths_string = "-".join([str(length) for length in TARGET_LENGTHS])

per_base_model_prepared_dicts = dict()
for base_model_name, base_model in per_name_base_model.items():
    per_allele_per_run_per_split_prepared_models = dict()
    for allele_name in ALLELES:
        per_allele_per_run_per_split_prepared_models[allele_name] = dict()
        for run_num in range(RUNS):
            print(f"Run {run_num}")
            per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
            for split_num, per_length_data in per_allele_per_kfold_per_length_immunogenic_train[allele_name].items():
                print(f"Split {split_num}")
                target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
                new_model_name = f"{run_num}_run_imm-{lengths_string}-mer-of-[{base_model_name}]_model-{target_allele_name}-{split_num}"

                base_model = per_name_base_model[base_model_name]

                new_model = base_model.copy()
                new_model.name = new_model_name

                per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = new_model
    per_base_model_prepared_dicts[base_model_name] = per_allele_per_run_per_split_prepared_models









Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4
Run 0
Split 0
Split 1
Split 2
Split 3
Split 4


In [14]:
keys = list(per_base_model_prepared_dicts.keys())
keys

['0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-0',
 '0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-1',
 '0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-2',
 '0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-3',
 '0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-4',
 '1_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-0',
 '1_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-1',
 '1_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-2',
 '1_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-3',
 '1_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-4',
 '1_run-9-18-st-baum-welch-alg-30-min_iters-69-max_iters_model-HLA_A_02_01-2',
 '1_run-9-18-st-baum-welch-alg-30-min_iters-69-max_iters_model-HLA_A_02_01-0',
 '0_run-9-18-st-baum-welch-alg-30-min_iters-69-max_iters_model-HLA_A_02_01-3']

### 3.1.2. Train prepared models

In [15]:
algorithm = 'baum-welch'
per_run_examples = dict()
for base_model_name in per_name_base_model:
    per_allele_models = dict()
    per_allele_histories = dict()
    per_allele_per_run_per_split_prepared_models = per_base_model_prepared_dicts[base_model_name]
    print(f"Now base model is {base_model_name}")
    for run_num in range(RUNS):
        print(f"run {run_num}")
        for allele_name in ALLELES:
            per_kfold_per_length_data =  per_allele_per_kfold_per_length_immunogenic_train[allele_name]
            models, histories = train_model(allele_name,
                                            per_kfold_per_length_data,
                                            TARGET_LENGTHS,
                                            verbose=False,
                                            multiple_check_input=False,
                                            algorithm=algorithm,
                                            prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                            stop_threshold=stop_threshold
                                            )
            per_allele_models[allele_name] = models
            per_allele_histories[allele_name] = histories
            per_run_examples[run_num] = models[0]
            # save models
            for split_num, model in per_allele_models[allele_name].items():
                history = per_allele_histories[allele_name][split_num]
                save_model(TARGET_PATH_TO_RESULTS, model,history )

Now base model is 0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-0
run 0
split 0
train size 398
Shuffling for fairness 
Simple training: Train Length 398, init_batch_size None
split 1
train size 398
Shuffling for fairness 
Simple training: Train Length 398, init_batch_size None
split 2
train size 398
Shuffling for fairness 
Simple training: Train Length 398, init_batch_size None
split 3
train size 399
Shuffling for fairness 
Simple training: Train Length 399, init_batch_size None
split 4
train size 399
Shuffling for fairness 
Simple training: Train Length 399, init_batch_size None
Now base model is 0_run-9-18-st-baum-welch-alg-30-min_iters_model-HLA_A_02_01-1
run 0
split 0
train size 398
Shuffling for fairness 
Simple training: Train Length 398, init_batch_size None
split 1
train size 398
Shuffling for fairness 
Simple training: Train Length 398, init_batch_size None
split 2
train size 398
Shuffling for fairness 
Simple training: Train Length 398, init_batch_size None
spli